# Logic-Zero · Task 9: DPO training (Option B — reduced scale)

Continues training the SFT LoRA adapter on (chosen, rejected) preference pairs sampled from the SFT model itself.

**Two-step pipeline:**
1. **Generate DPO data** (`data/gen_dpo_data.py`) — sample 4 responses per puzzle from SFT model, keep puzzles with at least one correct + one incorrect. **~6-8h on A100**, **resume-safe** (incremental write + Drive backup).
2. **Train DPO** (`train/dpo.py`) — TRL DPOTrainer on the pairs. **~1-2h on A100**.

**Per-bucket targets** (Option B — halved from spec for single-session feasibility):
```
PER_N_RAW       = {2: 30, 3: 100, 4: 200, 5: 200, 6: 200}  # raw puzzles attempted per bucket
MIN_PAIRS_PER_N = {2:  5, 3:  50, 4:  50, 5:  50, 6:  50}  # accepted-pair floor per bucket
```

**Required:** GPU (A100/L4/T4), SFT checkpoint already saved at `/content/drive/MyDrive/logic-zero/checkpoints/sft/best/`.

**Resume:** any partial `dpo_data.jsonl` on Drive is automatically restored at start.


## 1. Verify GPU + clean state

In [ ]:
import torch, gc
gc.collect()
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU — Runtime → Change runtime type → GPU')
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
assert free / 1e9 > 10, 'Less than 10GB free — restart runtime first'

## 2. Pull latest repo

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

## 3. Install dependencies

In [ ]:
import subprocess
def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(['pip', 'install', '-q', '--default-timeout=180'] + specs,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed: {specs}')
pip_inst(['trl>=0.13.0,<0.15.0', 'peft>=0.13.0,<0.16.0', 'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv', 'wandb'])
import transformers, peft, trl, z3
print(f'transformers={transformers.__version__}  peft={peft.__version__}  trl={trl.__version__}')

## 4. Secrets (WANDB optional — enables loss curve logging)

In [ ]:
from google.colab import userdata
for key in ['WANDB_API_KEY', 'HF_TOKEN']:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'✓ {key} loaded')
    except Exception:
        print(f'⚠ {key} not in Colab Secrets (optional)')
if 'WANDB_API_KEY' not in os.environ:
    os.environ['WANDB_DISABLED'] = 'true'
    print('wandb disabled (no key — training still runs)')

## 5. Mount Drive (REQUIRED — for SFT checkpoint + DPO data backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/logic-zero'
!mkdir -p {DRIVE_ROOT}/checkpoints/sft {DRIVE_ROOT}/checkpoints/dpo {DRIVE_ROOT}/data
assert os.path.isdir(DRIVE_ROOT), 'Drive mount failed'
print(f'✓ Drive root: {DRIVE_ROOT}')

## 6. Restore SFT checkpoint from Drive

DPO needs the SFT LoRA adapter to start from. If `results/checkpoints/sft/best/` is missing locally, copy from Drive.

In [ ]:
import shutil
sft_local = 'results/checkpoints/sft/best'
sft_drive = f'{DRIVE_ROOT}/checkpoints/sft/best'
if not os.path.isdir(sft_local):
    assert os.path.isdir(sft_drive), f'NO SFT checkpoint in Drive: {sft_drive}'
    os.makedirs(os.path.dirname(sft_local), exist_ok=True)
    shutil.copytree(sft_drive, sft_local)
    print(f'✓ Restored SFT checkpoint: {sft_local}')
adapter_file = f'{sft_local}/adapter_model.safetensors'
size_mb = os.path.getsize(adapter_file) / 1e6
assert size_mb > 10, f'SFT adapter too small ({size_mb}MB) — likely corrupted'
print(f'SFT adapter: {size_mb:.1f}MB  files: {sorted(os.listdir(sft_local))}')

## 7. Generate DPO preference pairs

Runs `data/gen_dpo_data.py`. **Resume-safe** — every accepted pair is appended to `data/dpo_data.jsonl` immediately, and the file is mirrored to Drive every 10 pairs. If Colab disconnects, re-running this cell picks up where it left off.

**Expected duration:** 6-8h on A100. Progress every 10 pairs.

**Tip:** open browser DevTools (F12) → Console, paste this to prevent idle disconnect:
```javascript
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
```


In [ ]:
DRIVE_DATA = f'{DRIVE_ROOT}/data'
!mkdir -p data logs
import time
print(f'Started at: {time.strftime("%Y-%m-%d %H:%M:%S")}', flush=True)
!python -u -m data.gen_dpo_data \
    --sft-adapter results/checkpoints/sft/best \
    --out data/dpo_data.jsonl \
    --drive-backup-dir {DRIVE_DATA} \
    --max-rounds 3 \
    2>&1 | tee logs/dpo_gen.log

## 8. Verify DPO data

In [ ]:
import json
from collections import Counter
with open('data/dpo_data.jsonl') as f:
    recs = [json.loads(l) for l in f]
print(f'Total pairs: {len(recs)}')
print(f'Per-n distribution: {dict(sorted(Counter(r["n"] for r in recs).items()))}')
print(f'\nMinimum required per spec:')
for n, m in {2:5, 3:50, 4:50, 5:50, 6:50}.items():
    cnt = sum(1 for r in recs if r['n'] == n)
    flag = '✓' if cnt >= m else '✗'
    print(f'  n={n}: {cnt}/{m} {flag}')
# Sanity-check one record
if recs:
    r = recs[0]
    print(f'\nSample record (n={r["n"]}):')
    print(f'  prompt:   {r["prompt"][:100]}...')
    print(f'  chosen:   {r["chosen"][:80]}...')
    print(f'  rejected: {r["rejected"][:80]}...')

## 9. Run DPO training

Default config: 2 epochs, batch=2 × grad-accum=4 (effective 8), LR=5e-6, β=0.1.

**Expected duration:** ~1-2h on A100 for ~500 pairs × 2 epochs. Final checkpoint saved to `results/checkpoints/dpo/best/` and auto-mirrored to Drive.

In [ ]:
DRIVE_DPO = f'{DRIVE_ROOT}/checkpoints/dpo'
print(f'Started at: {time.strftime("%Y-%m-%d %H:%M:%S")}', flush=True)
!python -u -m train.dpo \
    --sft-adapter results/checkpoints/sft/best \
    --dpo-data data/dpo_data.jsonl \
    --out-dir results/checkpoints/dpo \
    --epochs 2 \
    --batch-size 2 --grad-accum 4 \
    --lr 5e-6 --beta 0.1 \
    --drive-backup-dir {DRIVE_DPO} \
    2>&1 | tee logs/dpo_train.log

## 10. Verify DPO checkpoint

In [ ]:
dpo_local = 'results/checkpoints/dpo/best'
dpo_drive = f'{DRIVE_DPO}/best'
assert os.path.isdir(dpo_local), f'DPO training failed — no checkpoint at {dpo_local}'
files = sorted(os.listdir(dpo_local))
size_mb = os.path.getsize(f'{dpo_local}/adapter_model.safetensors') / 1e6
print(f'Local DPO checkpoint: {size_mb:.1f}MB, files={files}')
if os.path.isdir(dpo_drive):
    drive_size = os.path.getsize(f'{dpo_drive}/adapter_model.safetensors') / 1e6
    print(f'✓ Drive backup OK: {drive_size:.1f}MB at {dpo_drive}')
else:
    print(f'⚠ Drive backup missing — run: !cp -r {dpo_local} {dpo_drive}')

## 11. Smoke test: 1 puzzle through DPO model

Quick sanity check that the DPO-trained model loads and produces formatted output.

In [ ]:
from peft import PeftModel
from train.common import load_base_model, to_chat, extract_answer

# Free GPU first (training left big buffers)
import gc, torch
gc.collect()
torch.cuda.empty_cache()

model, tok = load_base_model()
model = model.to('cuda')
# Load the DPO-trained adapter (it's a continuation of SFT; same shape)
model = PeftModel.from_pretrained(model, dpo_local)
model.eval()
model.config.use_cache = True

dev = [json.loads(l) for l in open('data/dev_data.jsonl')]
rec = dev[5]
print('PUZZLE:\n' + rec['puzzle'][:300])
print('\nGT:', rec['ground_truth'])

with torch.no_grad():
    inp = tok(to_chat(tok, rec['puzzle']), return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=600, do_sample=False,
                          pad_token_id=tok.eos_token_id,
                          stop_strings=['</answer>'], tokenizer=tok,
                          temperature=None, top_p=None, top_k=None)
resp = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
print('\nRESPONSE:\n' + resp)
print(f'\nextracted: {extract_answer(resp, n=len(rec["ground_truth"]))}')

## ✅ Task 9 done

**Outputs:**
- `results/checkpoints/dpo/best/` — DPO LoRA adapter (local + Drive backup)
- `data/dpo_data.jsonl` — preference pairs used
- `logs/dpo_train.log` — full training log

### Next: Task 10 — re-run `02_eval.ipynb` with DPO adapter

In `02_eval.ipynb` cell-2, change:
```python
MODEL_KIND = 'dpo'  # was 'sft'
```
and update cell-10/cell-12 to point at `results/checkpoints/dpo/best` (or just add a new MODEL_KIND='dpo' branch).

Expected DPO vs SFT delta on eval set: **+4 to +9pp overall**, biggest gains on n=2/3/4.

**Decision criteria after eval:**
- DPO eval ≥ SFT eval + 4pp → success, pipeline works end-to-end
- DPO eval flat/worse → check β (try 0.05 or 0.2), inspect chosen/rejected quality